# 第145章: グリッドワールドの知能体 — 学習した世界モデルで計画する

## Grid World Agent Capstone

**難易度**: ★★★★★ | **所要時間**: 300-360分

---

### 学習目標

1. 簡単なグリッドワールド環境を自作し、観測・行動・報酬の設計を理解する
2. 観測画像をエンコードし、潜在空間で遷移を予測する**世界モデル**を構築する
3. 学習した世界モデルの中で **Model Predictive Control (MPC)** により計画を立てる
4. モデルベースエージェントとモデルフリー（Q学習）のサンプル効率を比較する
5. 「夢の中で想像する」軌跡を可視化し、世界モデルの精度を評価する

### 前提知識

| ノートブック | トピック |
|---|---|
| 142 | Model-Based RLの基礎 |
| 143 | DreamerV3 |
| 144 | Genie — インタラクティブな世界生成 |
| 75 | 強化学習入門 |
| 112 | 表現学習の基礎 |

> **注意**: このノートブックは Phase 7 (World Models) のキャップストーン課題です。
> すべてのコードは自己完結型で、GPU がなくても CPU で実行できます。

## 0. 環境セットアップ

必要なライブラリのインポートとデバイス選択を行います。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

# --- 再現性のためのシード固定 ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# --- デバイス選択 (MPS / CUDA / CPU) ---
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'使用デバイス: {device}')

# --- 日本語フォント設定 ---
try:
except ImportError:
    try:
        from matplotlib import font_manager
        jp_fonts = [f.name for f in font_manager.fontManager.ttflist
                    if 'Gothic' in f.name or 'Hiragino' in f.name or 'Noto' in f.name]
        if jp_fonts:
            plt.rcParams['font.family'] = jp_fonts[0]
            print(f'日本語フォント: {jp_fonts[0]}')
        else:
            print('警告: 日本語フォントが見つかりません。文字化けの可能性があります。')
    except Exception:
        print('警告: 日本語フォント設定に失敗しました。')

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True
print('セットアップ完了!')

---
## 1. KeyDoorGridWorld 環境

7×7 のグリッドワールドを作成します。

| 要素 | 説明 |
|---|---|
| **壁 (Wall)** | 通過不可のセル（灰色） |
| **鍵 (Key)** | エージェントが拾うとドアを開けられる（黄色） |
| **ドア (Door)** | 鍵を持っていると通過できる（茶色） |
| **ゴール (Goal)** | 鍵を持って到達すると +1 報酬（緑） |
| **エージェント** | 上下左右に移動できる（青） |

### 行動空間
- 0: 上、1: 下、2: 左、3: 右（離散4行動）

### 観測空間
- 7×7×3 の RGB ピクセル画像（各タイルを色で表現）

### 報酬設計
- ゴール到達（鍵あり）: **+1.0**
- 鍵を拾う: **+0.5**
- ステップペナルティ: **-0.01**

### 終了条件
- 鍵を持ってゴールに到達、または最大100ステップ

In [ ]:
class KeyDoorGridWorld:
    EMPTY = 0
    WALL = 1
    KEY = 2
    DOOR = 3
    GOAL = 4
    AGENT = 5

    # 行動: 上, 下, 左, 右
    ACTION_NAMES = ['上', '下', '左', '右']
    DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    # 色マップ (RGB, 0-1)
    COLORS = {
        0: [1.0, 1.0, 1.0],   # 空白 - 白
        1: [0.3, 0.3, 0.3],   # 壁 - 灰
        2: [1.0, 0.85, 0.0],  # 鍵 - 黄
        3: [0.6, 0.3, 0.1],   # ドア - 茶
        4: [0.0, 0.8, 0.2],   # ゴール - 緑
        5: [0.2, 0.4, 1.0],   # エージェント - 青
    }

    def __init__(self, max_steps=100):
        self.size = 7
        self.max_steps = max_steps
        self.n_actions = 4
        self._build_map()
        self.reset()

    def _build_map(self):
        self.base_grid = np.zeros((self.size, self.size), dtype=np.int32)
        # 外壁
        self.base_grid[0, :] = self.WALL
        self.base_grid[-1, :] = self.WALL
        self.base_grid[:, 0] = self.WALL
        self.base_grid[:, -1] = self.WALL
        # 内壁（仕切り）
        self.base_grid[1, 3] = self.WALL
        self.base_grid[2, 3] = self.WALL
        self.base_grid[3, 3] = self.WALL
        # ドア（壁の隙間）
        self.base_grid[4, 3] = self.DOOR
        # 鍵の位置
        self.key_pos = (2, 1)
        # ゴールの位置
        self.goal_pos = (2, 5)
        # エージェント初期位置
        self.start_pos = (5, 1)

    def reset(self):
        self.grid = self.base_grid.copy()
        self.grid[self.key_pos] = self.KEY
        self.grid[self.goal_pos] = self.GOAL
        self.agent_pos = self.start_pos
        self.has_key = False
        self.steps = 0
        self.done = False
        return self._get_obs()

    def _get_obs(self):
        obs = np.zeros((self.size, self.size, 3), dtype=np.float32)
        for r in range(self.size):
            for c in range(self.size):
                obs[r, c] = self.COLORS[self.grid[r, c]]
        # エージェントを描画
        obs[self.agent_pos[0], self.agent_pos[1]] = self.COLORS[self.AGENT]
        return obs

    def step(self, action):
        if self.done:
            return self._get_obs(), 0.0, True, {}

        self.steps += 1
        dr, dc = self.DELTAS[action]
        nr, nc = self.agent_pos[0] + dr, self.agent_pos[1] + dc
        reward = -0.01  # ステップペナルティ

        # 境界チェック
        if nr < 0 or nr >= self.size or nc < 0 or nc >= self.size:
            nr, nc = self.agent_pos

        target = self.grid[nr, nc]
        can_move = True
        if target == self.WALL:
            can_move = False
        elif target == self.DOOR and not self.has_key:
            can_move = False

        if can_move:
            self.agent_pos = (nr, nc)
            # 鍵を拾う
            if target == self.KEY:
                self.has_key = True
                self.grid[nr, nc] = self.EMPTY
                reward += 0.5
            # ゴール到達
            elif target == self.GOAL and self.has_key:
                reward += 1.0
                self.done = True

        # 最大ステップ
        if self.steps >= self.max_steps:
            self.done = True

        info = {'has_key': self.has_key, 'steps': self.steps}
        return self._get_obs(), reward, self.done, info

# テスト
env = KeyDoorGridWorld()
obs = env.reset()
print(f'観測の形状: {obs.shape}')
print(f'行動数: {env.n_actions}')
print(f'エージェント初期位置: {env.agent_pos}')
print(f'鍵の位置: {env.key_pos}')
print(f'ゴールの位置: {env.goal_pos}')

### 1.1 グリッドワールドの可視化

環境の初期状態を描画し、各要素の配置を確認します。

In [ ]:
def render_grid(env, title='グリッドワールド'):
    obs = env._get_obs()
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(obs, interpolation='nearest')

    # グリッド線
    for i in range(env.size + 1):
        ax.axhline(i - 0.5, color='black', linewidth=0.5)
        ax.axvline(i - 0.5, color='black', linewidth=0.5)

    # ラベル
    labels = {
        env.WALL: 'W', env.KEY: 'K', env.DOOR: 'D', env.GOAL: 'G'
    }
    for r in range(env.size):
        for c in range(env.size):
            if (r, c) == env.agent_pos:
                ax.text(c, r, 'A', ha='center', va='center',
                        fontsize=14, fontweight='bold', color='white')
            elif env.grid[r, c] in labels:
                ax.text(c, r, labels[env.grid[r, c]], ha='center', va='center',
                        fontsize=12, fontweight='bold')

    # 凡例
    legend_items = [
        mpatches.Patch(color=[0.2, 0.4, 1.0], label='エージェント (A)'),
        mpatches.Patch(color=[1.0, 0.85, 0.0], label='鍵 (K)'),
        mpatches.Patch(color=[0.6, 0.3, 0.1], label='ドア (D)'),
        mpatches.Patch(color=[0.0, 0.8, 0.2], label='ゴール (G)'),
        mpatches.Patch(color=[0.3, 0.3, 0.3], label='壁 (W)'),
    ]
    ax.legend(handles=legend_items, loc='upper left', bbox_to_anchor=(1, 1))
    ax.set_title(title, fontsize=14)
    ax.set_xticks(range(env.size))
    ax.set_yticks(range(env.size))
    plt.tight_layout()
    plt.show()

env = KeyDoorGridWorld()
env.reset()
render_grid(env, '初期状態のグリッドワールド')

---
## 2. 観測エンコーダ (ObservationEncoder)

7×7×3 の RGB 画像を **64次元の潜在ベクトル** に圧縮します。

```
入力: (B, 3, 7, 7) -> Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear -> 64次元
```

小さなグリッドなので軽量な CNN で十分です。

In [ ]:
class ObservationEncoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, latent_dim)

    def forward(self, x):
        # x: (B, 3, 7, 7)
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = h.reshape(h.size(0), -1)  # (B, 32*7*7)
        return self.fc(h)  # (B, 64)

# テスト
encoder = ObservationEncoder().to(device)
dummy_obs = torch.randn(4, 3, 7, 7).to(device)
z = encoder(dummy_obs)
print(f'入力形状:  {dummy_obs.shape}')
print(f'潜在形状:  {z.shape}')
print(f'パラメータ数: {sum(p.numel() for p in encoder.parameters()):,}')

---
## 3. 遷移モデル (TransitionModel)

現在の潜在状態 $z_t$ と行動 $a_t$ から、次の潜在状態 $\hat{z}_{t+1}$ を予測します。

$$\hat{z}_{t+1} = f_{\theta}(z_t, a_t)$$

行動は one-hot エンコーディング（4次元）で表現し、潜在ベクトルと結合します。

In [ ]:
class TransitionModel(nn.Module):
    def __init__(self, latent_dim=64, n_actions=4, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + n_actions, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.n_actions = n_actions

    def forward(self, z, action):
        # action: (B,) integer -> one-hot (B, n_actions)
        a_onehot = F.one_hot(action.long(), self.n_actions).float()
        inp = torch.cat([z, a_onehot], dim=-1)
        return self.net(inp)

# テスト
trans = TransitionModel().to(device)
dummy_a = torch.randint(0, 4, (4,)).to(device)
z_next = trans(z, dummy_a)
print(f'潜在形状:      {z.shape}')
print(f'行動形状:      {dummy_a.shape}')
print(f'次の潜在形状:  {z_next.shape}')
print(f'パラメータ数:  {sum(p.numel() for p in trans.parameters()):,}')

---
## 4. 報酬モデル (RewardModel)

潜在状態と行動から報酬を予測します。

$$\hat{r}_t = g_{\theta}(z_t, a_t)$$

これにより、世界モデル内で「想像」するだけで報酬を見積もれます。

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, latent_dim=64, n_actions=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + n_actions, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.n_actions = n_actions

    def forward(self, z, action):
        a_onehot = F.one_hot(action.long(), self.n_actions).float()
        inp = torch.cat([z, a_onehot], dim=-1)
        return self.net(inp).squeeze(-1)  # (B,)

# テスト
rew_model = RewardModel().to(device)
pred_r = rew_model(z, dummy_a)
print(f'予測報酬形状: {pred_r.shape}')
print(f'予測報酬の例: {pred_r.detach().cpu().numpy()}')

---
## 5. 観測デコーダ (ObservationDecoder)

エンコーダの逆操作です。潜在ベクトルから元の7×7×3画像を復元します。

```
入力: (B, 64) -> Linear -> Reshape -> ConvTranspose2d -> ReLU -> ConvTranspose2d -> Sigmoid -> (B, 3, 7, 7)
```

これはデバッグと可視化のために使用します — 世界モデルが何を「見ている」かを確認できます。

In [ ]:
class ObservationDecoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 32 * 7 * 7)
        self.deconv1 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=1, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 3, kernel_size=3, stride=1, padding=1)

    def forward(self, z):
        h = F.relu(self.fc(z))
        h = h.reshape(-1, 32, 7, 7)
        h = F.relu(self.deconv1(h))
        return torch.sigmoid(self.deconv2(h))  # (B, 3, 7, 7)

# テスト
decoder = ObservationDecoder().to(device)
recon = decoder(z)
print(f'潜在形状:   {z.shape}')
print(f'復元形状:   {recon.shape}')
print(f'値の範囲:   [{recon.min().item():.3f}, {recon.max().item():.3f}]')
print(f'パラメータ数: {sum(p.numel() for p in decoder.parameters()):,}')

---
## 6. 世界モデル (WorldModel)

4つのコンポーネントを統合した世界モデルクラスです。

### 損失関数

$$\mathcal{L} = \underbrace{\|x_t - \hat{x}_t\|^2}_{\text{再構成損失}} + \underbrace{\|z_{t+1} - \hat{z}_{t+1}\|^2}_{\text{遷移損失}} + \underbrace{\|r_t - \hat{r}_t\|^2}_{\text{報酬損失}}$$

- **再構成損失**: エンコーダ + デコーダが観測を正しく表現できるか
- **遷移損失**: 次の潜在状態を正しく予測できるか
- **報酬損失**: 報酬を正しく予測できるか

In [ ]:
class WorldModel(nn.Module):
    def __init__(self, latent_dim=64, n_actions=4):
        super().__init__()
        self.encoder = ObservationEncoder(latent_dim)
        self.transition = TransitionModel(latent_dim, n_actions)
        self.reward_model = RewardModel(latent_dim, n_actions)
        self.decoder = ObservationDecoder(latent_dim)
        self.latent_dim = latent_dim
        self.n_actions = n_actions

    def encode(self, obs):
        if obs.dim() == 3:
            obs = obs.unsqueeze(0)
        if obs.shape[-1] == 3:
            obs = obs.permute(0, 3, 1, 2)
        return self.encoder(obs)

    def imagine_step(self, z, action):
        z_next = self.transition(z, action)
        r_pred = self.reward_model(z, action)
        return z_next, r_pred

    def decode(self, z):
        return self.decoder(z)

    def compute_loss(self, obs, actions, rewards, next_obs):
        # エンコード
        obs_t = obs.permute(0, 3, 1, 2)           # (B,3,7,7)
        next_obs_t = next_obs.permute(0, 3, 1, 2)  # (B,3,7,7)

        z = self.encoder(obs_t)
        z_next_true = self.encoder(next_obs_t).detach()  # ターゲット

        # 再構成損失
        recon = self.decoder(z)
        loss_recon = F.mse_loss(recon, obs_t)

        # 遷移損失
        z_next_pred = self.transition(z, actions)
        loss_trans = F.mse_loss(z_next_pred, z_next_true)

        # 報酬損失
        r_pred = self.reward_model(z, actions)
        loss_reward = F.mse_loss(r_pred, rewards)

        total = loss_recon + loss_trans + loss_reward
        return total, loss_recon, loss_trans, loss_reward

# テスト
world_model = WorldModel().to(device)
n_params = sum(p.numel() for p in world_model.parameters())
print(f'世界モデル パラメータ総数: {n_params:,}')

---
## 7. ランダム探索によるデータ収集

まずはランダム方策で環境を探索し、遷移データを収集します。

収集するデータ: $(s_t, a_t, r_t, s_{t+1})$ の4つ組を **5,000件** 集めます。

これが世界モデルの学習データとなります。

In [ ]:
def collect_random_data(env, n_transitions=5000):
    observations = []
    actions = []
    rewards = []
    next_observations = []

    obs = env.reset()
    collected = 0

    while collected < n_transitions:
        action = np.random.randint(env.n_actions)
        next_obs, reward, done, _ = env.step(action)

        observations.append(obs.copy())
        actions.append(action)
        rewards.append(reward)
        next_observations.append(next_obs.copy())
        collected += 1

        if done:
            obs = env.reset()
        else:
            obs = next_obs

    return {
        'obs': np.array(observations),
        'actions': np.array(actions),
        'rewards': np.array(rewards),
        'next_obs': np.array(next_observations),
    }

env = KeyDoorGridWorld()
data = collect_random_data(env, n_transitions=5000)
print(f'収集データ数: {len(data["obs"])}')
print(f'観測形状:     {data["obs"].shape}')
print(f'行動の分布:   {np.bincount(data["actions"])}')
print(f'報酬の統計:   平均={data["rewards"].mean():.4f}, 最大={data["rewards"].max():.2f}')

---
## 8. 世界モデルの学習

収集したデータで世界モデルを学習します。

- **エポック数**: 50
- **バッチサイズ**: 256
- **学習率**: 1e-3
- **オプティマイザ**: Adam

In [ ]:
def train_world_model(model, data, n_epochs=50, batch_size=256, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n_samples = len(data['obs'])
    history = {'total': [], 'recon': [], 'trans': [], 'reward': []}

    obs_tensor = torch.FloatTensor(data['obs']).to(device)
    act_tensor = torch.LongTensor(data['actions']).to(device)
    rew_tensor = torch.FloatTensor(data['rewards']).to(device)
    nobs_tensor = torch.FloatTensor(data['next_obs']).to(device)

    for epoch in range(n_epochs):
        indices = np.random.permutation(n_samples)
        epoch_losses = {'total': 0, 'recon': 0, 'trans': 0, 'reward': 0}
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_obs = obs_tensor[idx]
            batch_act = act_tensor[idx]
            batch_rew = rew_tensor[idx]
            batch_nobs = nobs_tensor[idx]

            loss_total, loss_r, loss_t, loss_rw = model.compute_loss(
                batch_obs, batch_act, batch_rew, batch_nobs
            )

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

            epoch_losses['total'] += loss_total.item()
            epoch_losses['recon'] += loss_r.item()
            epoch_losses['trans'] += loss_t.item()
            epoch_losses['reward'] += loss_rw.item()
            n_batches += 1

        for k in epoch_losses:
            epoch_losses[k] /= n_batches
            history[k].append(epoch_losses[k])

        if (epoch + 1) % 10 == 0:
            print(f'エポック {epoch+1:3d}/{n_epochs} | '
                  f'合計: {epoch_losses["total"]:.4f} | '
                  f'再構成: {epoch_losses["recon"]:.4f} | '
                  f'遷移: {epoch_losses["trans"]:.4f} | '
                  f'報酬: {epoch_losses["reward"]:.4f}')

    return history

world_model = WorldModel().to(device)
history = train_world_model(world_model, data, n_epochs=50, batch_size=256, lr=1e-3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ['合計損失', '再構成損失', '遷移損失', '報酬損失']
keys = ['total', 'recon', 'trans', 'reward']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for ax, title, key, color in zip(axes, titles, keys, colors):
    ax.plot(history[key], color=color, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('エポック')
    ax.set_ylabel('損失')

fig.suptitle('世界モデルの学習曲線', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. 学習した表現の可視化

エンコーダ→デコーダで再構成した画像と元の画像を比較します。
また、遷移モデルで予測した次状態と実際の次状態を比較します。

In [ ]:
world_model.eval()
sample_idx = np.random.choice(len(data['obs']), 5, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
with torch.no_grad():
    for i, idx in enumerate(sample_idx):
        # 元の観測
        orig = data['obs'][idx]
        axes[0, i].imshow(orig, interpolation='nearest')
        axes[0, i].set_title(f'元の画像 #{idx}')

        # 再構成
        obs_t = torch.FloatTensor(orig).unsqueeze(0).permute(0, 3, 1, 2).to(device)
        z = world_model.encoder(obs_t)
        recon = world_model.decoder(z).cpu().squeeze(0).permute(1, 2, 0).numpy()
        recon = np.clip(recon, 0, 1)
        axes[1, i].imshow(recon, interpolation='nearest')
        axes[1, i].set_title('再構成画像')

        # 遷移予測
        action = torch.LongTensor([data['actions'][idx]]).to(device)
        z_next = world_model.transition(z, action)
        pred_next = world_model.decoder(z_next).cpu().squeeze(0).permute(1, 2, 0).numpy()
        pred_next = np.clip(pred_next, 0, 1)
        axes[2, i].imshow(pred_next, interpolation='nearest')
        act_name = KeyDoorGridWorld.ACTION_NAMES[data['actions'][idx]]
        axes[2, i].set_title(f'予測次状態 ({act_name})')

for ax_row in axes:
    for ax in ax_row:
        ax.set_xticks([])
        ax.set_yticks([])

axes[0, 0].set_ylabel('元の画像', fontsize=12)
axes[1, 0].set_ylabel('再構成', fontsize=12)
axes[2, 0].set_ylabel('遷移予測', fontsize=12)
fig.suptitle('世界モデルの再構成と遷移予測', fontsize=15)
plt.tight_layout()
plt.show()

---
## 10. Model Predictive Control (MPC)

学習した世界モデルの中で「想像」することで計画を立てます。

### アルゴリズム

1. 現在の観測を潜在ベクトル $z_t$ にエンコード
2. $K=50$ 個のランダムな行動列を生成（各長さ $H=10$）
3. 各行動列について、世界モデル内で $H$ ステップ先まで「想像」
4. 累積予測報酬が最大の行動列を選択
5. その列の**最初の行動のみ**を実行
6. 次のステップで再計画（リシーディングホライゾン）

これは **shooting method** と呼ばれる単純なMPCです。

In [ ]:
class ModelPredictiveControl:
    def __init__(self, world_model, n_candidates=50, horizon=10):
        self.model = world_model
        self.n_candidates = n_candidates
        self.horizon = horizon

    @torch.no_grad()
    def plan(self, obs):
        self.model.eval()

        # 現在の状態をエンコード
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        z0 = self.model.encode(obs_t)  # (1, 64)

        K = self.n_candidates
        H = self.horizon
        n_actions = self.model.n_actions

        action_seqs = torch.randint(0, n_actions, (K, H)).to(device)
        total_rewards = torch.zeros(K).to(device)

        # 各候補について想像
        z = z0.expand(K, -1).clone()  # (K, 64)

        for t in range(H):
            actions_t = action_seqs[:, t]
            z_next, r_pred = self.model.imagine_step(z, actions_t)
            total_rewards += r_pred
            z = z_next

        # 最良の候補を選択
        best_idx = total_rewards.argmax()
        best_action = action_seqs[best_idx, 0].item()
        return best_action

# テスト
mpc = ModelPredictiveControl(world_model, n_candidates=50, horizon=10)
env = KeyDoorGridWorld()
obs = env.reset()
action = mpc.plan(obs)
print(f'計画された行動: {action} ({KeyDoorGridWorld.ACTION_NAMES[action]})')

---
## 11. 世界モデルエージェント (WorldModelAgent)

4つのフェーズを反復するエージェントです:

| フェーズ | 説明 | ステップ数 |
|---|---|---|
| 1. 探索 | ランダム方策でデータ収集 | 500 |
| 2. 学習 | 世界モデルを学習 | 50エポック |
| 3. 計画 | MPCで行動を計画 | - |
| 4. 実行 | 計画した行動を環境で実行、データ追加 | - |

フェーズ3-4を繰り返しながら、定期的にフェーズ2で世界モデルを更新します。

In [ ]:
class WorldModelAgent:
    def __init__(self, env, latent_dim=64, mpc_candidates=50, mpc_horizon=10):
        self.env = env
        self.world_model = WorldModel(latent_dim, env.n_actions).to(device)
        self.mpc = ModelPredictiveControl(self.world_model, mpc_candidates, mpc_horizon)
        self.buffer = {'obs': [], 'actions': [], 'rewards': [], 'next_obs': []}

    def add_to_buffer(self, obs, action, reward, next_obs):
        self.buffer['obs'].append(obs.copy())
        self.buffer['actions'].append(action)
        self.buffer['rewards'].append(reward)
        self.buffer['next_obs'].append(next_obs.copy())

    def get_buffer_data(self):
        return {
            'obs': np.array(self.buffer['obs']),
            'actions': np.array(self.buffer['actions']),
            'rewards': np.array(self.buffer['rewards']),
            'next_obs': np.array(self.buffer['next_obs']),
        }

    def explore(self, n_steps=500):
        obs = self.env.reset()
        for _ in range(n_steps):
            action = np.random.randint(self.env.n_actions)
            next_obs, reward, done, _ = self.env.step(action)
            self.add_to_buffer(obs, action, reward, next_obs)
            obs = next_obs if not done else self.env.reset()

    def learn(self, n_epochs=50, batch_size=256, lr=1e-3, verbose=False):
        data = self.get_buffer_data()
        optimizer = optim.Adam(self.world_model.parameters(), lr=lr)
        n_samples = len(data['obs'])
        obs_t = torch.FloatTensor(data['obs']).to(device)
        act_t = torch.LongTensor(data['actions']).to(device)
        rew_t = torch.FloatTensor(data['rewards']).to(device)
        nobs_t = torch.FloatTensor(data['next_obs']).to(device)

        self.world_model.train()
        for epoch in range(n_epochs):
            idx = np.random.permutation(n_samples)
            total_loss = 0
            n_b = 0
            for i in range(0, n_samples, batch_size):
                b = idx[i:i+batch_size]
                loss, _, _, _ = self.world_model.compute_loss(
                    obs_t[b], act_t[b], rew_t[b], nobs_t[b])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                n_b += 1
            if verbose and (epoch + 1) % 25 == 0:
                print(f'  学習エポック {epoch+1}/{n_epochs}: 損失={total_loss/n_b:.4f}')

    def run_episode(self, use_mpc=True, max_steps=100):
        obs = self.env.reset()
        total_reward = 0
        trajectory = [obs.copy()]

        for step in range(max_steps):
            if use_mpc:
                action = self.mpc.plan(obs)
            else:
                action = np.random.randint(self.env.n_actions)

            next_obs, reward, done, info = self.env.step(action)
            self.add_to_buffer(obs, action, reward, next_obs)
            total_reward += reward
            trajectory.append(next_obs.copy())
            obs = next_obs
            if done:
                break

        return total_reward, step + 1, trajectory, info

# テスト
agent = WorldModelAgent(KeyDoorGridWorld())
print('WorldModelAgent 作成完了')
print(f'パラメータ数: {sum(p.numel() for p in agent.world_model.parameters()):,}')

### 11.1 エージェントの実行

反復的な探索→学習→計画→実行のループを実行します。

In [ ]:
# --- 世界モデルエージェントの学習ループ ---
agent = WorldModelAgent(KeyDoorGridWorld())

# フェーズ1: 初期ランダム探索
print('=== フェーズ1: ランダム探索 (500ステップ) ===')
agent.explore(n_steps=500)
print(f'バッファサイズ: {len(agent.buffer["obs"])}')

# フェーズ2: 初期学習
print('\n=== フェーズ2: 世界モデル学習 (50エポック) ===')
agent.learn(n_epochs=50, verbose=True)

# フェーズ3-4: 計画と実行を反復
print('\n=== フェーズ3-4: 計画と実行 ===')
wm_rewards = []
wm_solved = []
n_episodes = 30
retrain_every = 10

for ep in range(n_episodes):
    reward, steps, traj, info = agent.run_episode(use_mpc=True)
    wm_rewards.append(reward)
    solved = reward > 0.5  # 鍵+ゴールで成功
    wm_solved.append(solved)

    if (ep + 1) % 5 == 0:
        print(f'エピソード {ep+1:3d} | 報酬: {reward:6.2f} | '
              f'ステップ: {steps:3d} | 解決: {solved}')

    # 定期的に再学習
    if (ep + 1) % retrain_every == 0:
        print(f'  >> 世界モデルを再学習 (バッファ: {len(agent.buffer["obs"])}件)')
        agent.learn(n_epochs=30, verbose=False)

print(f'\n世界モデルエージェント 解決率: {sum(wm_solved)}/{n_episodes}')
print(f'累積報酬: {sum(wm_rewards):.2f}')

---
## 12. Q学習ベースライン

モデルフリーの **表形式Q学習** を同じグリッドワールドで実行し、
サンプル効率を比較します。

状態: `(row, col, has_key)` の3つ組でテーブルを構築します。

In [ ]:
def q_learning_baseline(n_episodes=300, lr=0.1, gamma=0.99, epsilon=0.3, decay=0.995):
    env = KeyDoorGridWorld()
    q_table = {}

    def get_state(env):
        return (env.agent_pos[0], env.agent_pos[1], int(env.has_key))

    def get_q(state, action):
        return q_table.get((state, action), 0.0)

    rewards_history = []
    solved_history = []
    eps = epsilon

    for ep in range(n_episodes):
        env.reset()
        state = get_state(env)
        total_reward = 0

        for step in range(100):
            # epsilon-greedy
            if np.random.random() < eps:
                action = np.random.randint(4)
            else:
                q_vals = [get_q(state, a) for a in range(4)]
                action = np.argmax(q_vals)

            _, reward, done, _ = env.step(action)
            next_state = get_state(env)
            total_reward += reward

            # Q更新
            best_next = max(get_q(next_state, a) for a in range(4))
            old_q = get_q(state, action)
            q_table[(state, action)] = old_q + lr * (reward + gamma * best_next - old_q)

            state = next_state
            if done:
                break

        rewards_history.append(total_reward)
        solved_history.append(total_reward > 0.5)
        eps *= decay

    return rewards_history, solved_history

print('Q学習ベースラインを実行中...')
ql_rewards, ql_solved = q_learning_baseline(n_episodes=300)
print(f'Q学習 解決率 (最後の30エピソード): {sum(ql_solved[-30:])}/30')
print(f'Q学習 累積報酬 (最後の30エピソード): {sum(ql_rewards[-30:]):.2f}')

---
## 13. 比較: 世界モデル vs Q学習

サンプル効率を比較します。
世界モデルエージェントは少ない環境インタラクションで学習できるはずです。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 累積報酬の推移 ---
ax = axes[0]
wm_cum = np.cumsum(wm_rewards)
ax.plot(range(1, len(wm_rewards)+1), wm_cum, 'b-o', label='世界モデルエージェント',
        linewidth=2, markersize=4)
ql_cum_30 = np.cumsum(ql_rewards[:30])
ax.plot(range(1, 31), ql_cum_30, 'r-s', label='Q学習 (最初の30)',
        linewidth=2, markersize=4)
ax.set_xlabel('エピソード')
ax.set_ylabel('累積報酬')
ax.set_title('累積報酬の比較')
ax.legend()

# --- 移動平均報酬 ---
ax = axes[1]
window = 10
ql_ma = np.convolve(ql_rewards, np.ones(window)/window, mode='valid')
ax.plot(range(window, len(ql_rewards)+1), ql_ma, 'r-', label='Q学習', linewidth=2)
if len(wm_rewards) >= window:
    wm_ma = np.convolve(wm_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window, len(wm_rewards)+1), wm_ma, 'b-',
            label='世界モデル', linewidth=2)
ax.set_xlabel('エピソード')
ax.set_ylabel('平均報酬 (10エピソード移動平均)')
ax.set_title('報酬の学習曲線')
ax.legend()

# --- 解決までのエピソード数 ---
ax = axes[2]
wm_first = next((i+1 for i, s in enumerate(wm_solved) if s), len(wm_solved))
ql_first = next((i+1 for i, s in enumerate(ql_solved) if s), len(ql_solved))
bars = ax.bar(['世界モデル\nエージェント', 'Q学習'], [wm_first, ql_first],
              color=['#2196F3', '#E91E63'], width=0.5)
ax.set_ylabel('エピソード数')
ax.set_title('初回解決までのエピソード数')
for bar, val in zip(bars, [wm_first, ql_first]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=14, fontweight='bold')

fig.suptitle('モデルベース vs モデルフリー: サンプル効率の比較', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'世界モデルエージェント — 初回解決: エピソード {wm_first}')
print(f'Q学習 — 初回解決: エピソード {ql_first}')
print(f'\n世界モデルの環境インタラクション総数: {len(agent.buffer["obs"])}')

---
## 14. 夢の可視化 — 想像された軌跡

世界モデルの中で「夢を見る」ことで、実環境に触れずに軌跡を生成します。
この想像された軌跡と実際の軌跡を比較しましょう。

In [ ]:
@torch.no_grad()
def dream_trajectory(world_model, start_obs, action_sequence):
    world_model.eval()
    obs_t = torch.FloatTensor(start_obs).unsqueeze(0).to(device)
    z = world_model.encode(obs_t)

    dream_images = [world_model.decode(z).cpu().squeeze(0).permute(1, 2, 0).numpy()]
    dream_rewards = []

    for a in action_sequence:
        action = torch.LongTensor([a]).to(device)
        z, r = world_model.imagine_step(z, action)
        img = world_model.decode(z).cpu().squeeze(0).permute(1, 2, 0).numpy()
        dream_images.append(np.clip(img, 0, 1))
        dream_rewards.append(r.item())

    return dream_images, dream_rewards

def real_trajectory(env, action_sequence):
    obs = env.reset()
    real_images = [obs.copy()]
    real_rewards = []

    for a in action_sequence:
        obs, r, done, _ = env.step(a)
        real_images.append(obs.copy())
        real_rewards.append(r)
        if done:
            break

    return real_images, real_rewards

# 行動列を設定 (上x3, 左, 上) — 鍵を目指す動き
action_seq = [0, 0, 0, 2, 0]
env = KeyDoorGridWorld()
start_obs = env.reset()

dream_imgs, dream_rews = dream_trajectory(agent.world_model, start_obs, action_seq)
real_imgs, real_rews = real_trajectory(env, action_seq)

n_steps = min(len(dream_imgs), len(real_imgs))
fig, axes = plt.subplots(2, n_steps, figsize=(3 * n_steps, 6))

for t in range(n_steps):
    axes[0, t].imshow(real_imgs[t], interpolation='nearest')
    axes[0, t].set_title(f't={t}', fontsize=11)
    axes[0, t].set_xticks([])
    axes[0, t].set_yticks([])

    axes[1, t].imshow(dream_imgs[t], interpolation='nearest')
    axes[1, t].set_title(f't={t}', fontsize=11)
    axes[1, t].set_xticks([])
    axes[1, t].set_yticks([])

axes[0, 0].set_ylabel('実際の軌跡', fontsize=12)
axes[1, 0].set_ylabel('想像された軌跡', fontsize=12)
fig.suptitle('夢の可視化: 実際 vs 想像', fontsize=15)
plt.tight_layout()
plt.show()

print('行動列:', [KeyDoorGridWorld.ACTION_NAMES[a] for a in action_seq])
print(f'実際の報酬:   {real_rews}')
print(f'想像された報酬: {[f"{r:.3f}" for r in dream_rews]}')

In [ ]:
# より長い想像軌跡
long_action_seq = [0, 0, 0, 2, 0, 0, 3, 3, 3, 1, 1, 3, 3, 0, 0]
dream_imgs_long, dream_rews_long = dream_trajectory(
    agent.world_model, start_obs, long_action_seq)

n_show = min(10, len(dream_imgs_long))
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for t in range(n_show):
    axes[t].imshow(dream_imgs_long[t], interpolation='nearest')
    if t < len(long_action_seq):
        axes[t].set_title(f'{KeyDoorGridWorld.ACTION_NAMES[long_action_seq[t]]}', fontsize=10)
    else:
        axes[t].set_title('end', fontsize=10)
    axes[t].set_xticks([])
    axes[t].set_yticks([])

fig.suptitle('世界モデルの長期想像 (10ステップ)', fontsize=14)
plt.tight_layout()
plt.show()
print(f'想像された累積報酬: {sum(dream_rews_long[:n_show-1]):.3f}')

---
## 確認クイズ

### Q1. 世界モデルの3つの損失関数に含まれないのはどれか？

- (a) 再構成損失（観測の復元誤差）
- (b) 遷移損失（次状態の予測誤差）
- (c) 方策損失（行動確率の最適化）
- (d) 報酬損失（報酬の予測誤差）

### Q2. Model Predictive Control (MPC) の「リシーディングホライゾン」とは？

- (a) 計画のホライゾンを徐々に短くすること
- (b) 毎ステップ再計画し、最初の行動のみ実行すること
- (c) 複数のモデルを切り替えて使うこと
- (d) 報酬を割り引くこと

### Q3. モデルベースRLがモデルフリーRLより一般的にサンプル効率が良い理由は？

- (a) ニューラルネットワークが大きいため
- (b) 学習した世界モデル内で計画でき、実環境との相互作用を減らせるため
- (c) 報酬関数を手動で設計するため
- (d) 行動空間が連続だから

### Q4. 観測エンコーダの役割として最も適切なのは？

- (a) 行動を選択する
- (b) 高次元の観測を低次元の潜在表現に圧縮する
- (c) 報酬を計算する
- (d) 環境をリセットする

### Q5. 世界モデルの「夢」（想像軌跡）が実際の軌跡と異なる主な原因は？

- (a) 乱数シードが異なるため
- (b) モデルの予測誤差が累積するため（compounding error）
- (c) GPUの計算精度が低いため
- (d) 行動空間が連続だから

### クイズの答え

| 問題 | 答え | 解説 |
|---|---|---|
| Q1 | **(c)** | 世界モデルは環境のダイナミクスを学習するもので、方策損失は含まれません。方策最適化は別のステップです。 |
| Q2 | **(b)** | リシーディングホライゾンでは毎ステップで再計画し、最初の行動のみを実行します。 |
| Q3 | **(b)** | 世界モデル内で「想像」することで、実環境との相互作用を大幅に削減できます。 |
| Q4 | **(b)** | エンコーダは高次元の観測（画像）を計算しやすい低次元の潜在空間に圧縮します。 |
| Q5 | **(b)** | 予測誤差が各ステップで累積し、長い軌跡ほど実際との乖離が大きくなります。 |

---
## まとめ

このノートブックでは、Phase 7 (World Models) のキャップストーンとして、
**グリッドワールドの知能体** を一から構築しました。

### 学んだこと

| トピック | 内容 |
|---|---|
| 環境設計 | KeyDoorGridWorld: 壁・鍵・ドア・ゴールのある7x7グリッド |
| 観測エンコーダ | CNN で画像を64次元潜在ベクトルに圧縮 |
| 遷移モデル | MLP で次の潜在状態を予測 |
| 報酬モデル | MLP で報酬を予測 |
| 観測デコーダ | 潜在表現から画像を復元（デバッグ用） |
| 世界モデル | 4つのコンポーネントの統合、3つの損失関数 |
| MPC | ランダムシューティングによる計画 |
| エージェント | 探索→学習→計画→実行の4フェーズループ |
| 比較 | モデルベース vs モデルフリー（Q学習）のサンプル効率 |

### 前のノートブックとの関連

- **142 (Model-Based RL)**: 今回実装した世界モデルは、142章で学んだ概念の具体的な実装です
- **143 (DreamerV3)**: 「夢の中で学習する」というアイデアを簡略化して実装しました
- **144 (Genie)**: 世界モデルの生成能力を確認するデコーダは、Genieの核心概念に通じます
- **75 (強化学習入門)**: Q学習ベースラインとの比較で、基礎からの進歩を体感できました
- **112 (表現学習)**: 観測エンコーダは表現学習そのものです

### キーポイント

1. **世界モデル = 環境のシミュレータ**: 遷移と報酬を潜在空間で予測する
2. **想像で計画**: 実環境に触れずにモデル内で行動の帰結を評価できる
3. **サンプル効率**: モデルベースはモデルフリーより少ない相互作用で学習できる
4. **誤差の累積**: 長期予測ではモデル誤差が蓄積する（compounding error）
5. **探索と活用**: 良いデータがないとモデルも不正確になる

---
## 次のステップ

次のノートブック **146: 世界モデルの統合** では、
Phase 7 で学んだすべての概念を振り返り、統合的な理解を深めます。

### 発展課題（オプション）

1. **環境の拡張**: より大きなグリッド（10x10）、複数の鍵、複数のドアを追加してみましょう
2. **CEM (Cross-Entropy Method)**: ランダムシューティングの代わりに、より洗練された計画アルゴリズムを実装してみましょう
3. **VAEエンコーダ**: 決定的なエンコーダの代わりに、変分オートエンコーダを使ってみましょう
4. **Dreamer風学習**: 世界モデルの中で方策も同時に学習する方法を試してみましょう
5. **確率的遷移**: 遷移モデルをガウス分布にして不確実性を扱ってみましょう

---

*Phase 7: World Models — Complete!*